In [1]:
import math
from collections.abc import Iterable, Iterator, Buffer
from typing import Any, override, SupportsIndex, Self

import numpy as np
import numpy.typing as npt
from dnnlpy.models.mlp import Module, Optimizer
from numpy import _OrderKACF
from numpy._typing import _ShapeLike, DTypeLike

rng = np.random.default_rng(42)
print("Numpy version:", np.__version__)

Numpy version: 2.5.2


In [2]:
"""
线性层的前向传播
    线性层的作用就是将每个样本从D_in维空间映射到D_out维空间
"""

B = 4
D_in = 3
D_out = 2

x = rng.random((B, D_in))
w = rng.random((D_in, D_out))
b = np.zeros(D_out)

Y = x @ w + b

print('x.shape:', x.shape)
print('w.shape:', w.shape)
print('b.shape:', b.shape)
print('y.shape:', Y.shape)

x.shape: (4, 3)
w.shape: (3, 2)
b.shape: (2,)
y.shape: (4, 2)


In [3]:
"""
线性层的反向传播
"""

G = rng.random((B, D_out))

dX = G @ w.T
dW = x.T @ G
dB = np.sum(G, axis=0)

print('dx.shape', dX.shape)
print('dw.shape', dW.shape)
print('db.shape', dB.shape)

dx.shape (4, 3)
dw.shape (3, 2)
db.shape (2,)


In [10]:
"""
用numpy 实现Linear层
"""


class Parameter(np.ndarray):
    __array_priority__ = 1000
    grad: np.ndarray | None

    def __new__(
            cls,
            data: Any,
            dtype: npt.DTypeLike = np.float32
    ):
        obj = np.asarray(data, dtype=dtype).view(cls)
        obj.grad = None
        return obj

    def __array_finalize__(self, obj: Any):
        if obj is None:
            return
        self.grad = getattr(obj, 'grad', None)

    def __array_wrap__(self, out_arr: Any, context=None, return_scalar=False):
        return np.asarray(out_arr)

    @property
    def data(self) -> np.ndarray:
        return np.asarray(self)


class Linear(Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features

        weight = rng.standard_normal((in_features, out_features))
        weight = weight * math.sqrt(2.0 / in_features)
        bias = np.zeros(out_features)

        self.weight = Parameter(weight)
        self.bias = Parameter(bias)

    @override
    def forward(self, x: np.ndarray) -> np.ndarray:
        self.ctx = x
        return x @ self.weight + self.bias

    @override
    def backward(self, grad: np.ndarray) -> np.ndarray:
        assert self.ctx is not None, "Must Call  forward before backward."
        x = self.ctx

        self.weight.grad = x.T @ grad
        self.bias.grad = np.sum(grad, axis=0)
        return grad @ self.weight.T
    @override
    def parameters(self) -> Iterator[Parameter]:
        for value in self.__dict__.values():
            if isinstance(value, Parameter):
                yield value
            elif isinstance(value, Module):
                yield from value.parameters()


linear = Linear(in_features=3, out_features=2)
x = rng.random((4, 3))
out = linear(x)

dout = rng.random(out.shape)
dx = linear.backward(dout)

print("out.shape:", out.shape)
print("dx.shape:", dx.shape)
print("dW.shape:", linear.weight.grad.shape)
print("db.shape:", linear.bias.grad.shape)


out.shape: (4, 2)
dx.shape: (4, 3)
dW.shape: (3, 2)
db.shape: (2,)


In [13]:
class SGD(Optimizer):
    def __init__(self, params: Iterable[Parameter], lr: float = 1e-3):
        super().__init__(params)
        self.lr = lr
    @override
    def step(self):
        for p in self.params:
            if p.grad is None:
                continue
            p -= self.lr * p.grad

optimizer = SGD(linear.parameters(), lr=1e-3)

print("Original W:n",linear.weight)
print("Original b:n",linear.bias)

optimizer.step()

print("Updated W:n",linear.weight)
print("Updated b:n",linear.bias)



Original W:n [[-0.31827837 -1.1240596 ]
 [ 0.51859856 -0.18144408]
 [-1.2009083  -0.82921684]]
Original b:n [0. 0.]
Updated W:n [[-0.3191918  -1.1256945 ]
 [ 0.51824623 -0.18250671]
 [-1.2014327  -0.8302069 ]]
Updated b:n [-0.00135635 -0.00278186]
